# Lab 3: Entropy, KL Divergence, and Fitting Distributions

Information-theoretic quantities — entropy, cross-entropy, KL divergence — appear in loss functions, regularization terms, and training objectives throughout deep learning. This lab implements them from scratch, verifies their properties, and demonstrates the mode-seeking vs mean-seeking behavior that determines whether a model learns a sharp or blurry approximation.

**Sections**
1. Entropy
2. KL Divergence
3. Cross-Entropy Decomposition
4. Forward vs Reverse KL Fitting
5. Beta Posterior Updates

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats, optimize

plt.rcParams['figure.figsize'] = (10, 4)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
rng = np.random.default_rng(3)
print('Setup complete.')

## 1. Entropy

$H(p) = -\sum_x p(x)\log_2 p(x)$ bits.

The uniform distribution over $K$ outcomes maximizes entropy at $\log_2 K$ bits. For a Bernoulli, entropy peaks at $\theta=0.5$ (1 bit) and hits 0 at the degenerate extremes.

In [ ]:
def entropy_discrete(p):
    """Shannon entropy in bits. p is a probability array summing to 1."""
    p = np.asarray(p, dtype=float)
    mask = p > 0
    return -np.sum(p[mask] * np.log2(p[mask]))

# Bernoulli entropy vs bias
thetas = np.linspace(0.001, 0.999, 500)
h_bernoulli = [entropy_discrete([t, 1-t]) for t in thetas]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(thetas, h_bernoulli, 'steelblue', lw=2)
ax1.axvline(0.5, color='gray', linestyle=':', alpha=0.7)
ax1.axhline(1.0, color='coral', linestyle='--', alpha=0.7, label='max = 1 bit')
ax1.set_xlabel('θ = P(heads)')
ax1.set_ylabel('Entropy (bits)')
ax1.set_title('Bernoulli entropy vs bias')
ax1.legend()

# Entropy vs K for uniform distribution
Ks = np.arange(2, 513)
h_uniform = np.log2(Ks)
ax2.plot(Ks, h_uniform, 'steelblue', lw=2)
ax2.set_xlabel('K (number of outcomes)')
ax2.set_ylabel('Entropy (bits)')
ax2.set_title('Max entropy = log₂K for uniform over K outcomes')
for k, label in [(2,'coins'), (10,'digits'), (26,'letters'), (512,'vocab')]:
    ax2.scatter([k], [np.log2(k)], s=60, zorder=5)
    ax2.annotate(label, (k, np.log2(k)), textcoords='offset points', xytext=(5,3), fontsize=8)

plt.tight_layout()
plt.show()

# Spot checks
print(f"H(Bernoulli(0.5)) = {entropy_discrete([0.5, 0.5]):.4f} bits  (should be 1.0)")
print(f"H(Uniform over 8) = {entropy_discrete([1/8]*8):.4f} bits  (should be {np.log2(8):.4f})")
print(f"H(degenerate [1,0])= {entropy_discrete([1.0, 0.0]):.4f} bits  (should be 0.0)")

## 2. KL Divergence

$D_{\text{KL}}(p \| q) = \sum_x p(x)\log\frac{p(x)}{q(x)}$.

Properties: non-negative, zero iff $p=q$, asymmetric.

In [ ]:
def kl_divergence(p, q):
    """D_KL(p || q) for discrete distributions. q[i] > 0 wherever p[i] > 0."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return np.sum(p[mask] * np.log(p[mask] / q[mask]))

# Verify properties
p = np.array([0.5, 0.3, 0.2])
q = np.array([0.3, 0.4, 0.3])
uniform = np.array([1/3, 1/3, 1/3])

print("Non-negativity:")
print(f"  D_KL(p‖q) = {kl_divergence(p, q):.6f} ≥ 0  ✓")
print(f"  D_KL(q‖p) = {kl_divergence(q, p):.6f} ≥ 0  ✓")
print()
print("Identity (D_KL(p‖p) = 0):")
print(f"  D_KL(p‖p) = {kl_divergence(p, p):.10f}  ✓")
print()
print("Asymmetry (D_KL(p‖q) ≠ D_KL(q‖p)):")
print(f"  D_KL(p‖q) = {kl_divergence(p, q):.6f}")
print(f"  D_KL(q‖p) = {kl_divergence(q, p):.6f}  (different!)")

# Sweep: KL as a function of how different two Bernoulli distributions are
p_fixed = 0.3
q_range = np.linspace(0.01, 0.99, 300)
kl_pq = [kl_divergence([p_fixed, 1-p_fixed], [q_, 1-q_]) for q_ in q_range]
kl_qp = [kl_divergence([q_, 1-q_], [p_fixed, 1-p_fixed]) for q_ in q_range]

plt.figure(figsize=(9, 4))
plt.plot(q_range, kl_pq, 'steelblue', lw=2, label='D_KL(p‖q)  p fixed at 0.3')
plt.plot(q_range, kl_qp, 'coral', lw=2, linestyle='--', label='D_KL(q‖p)')
plt.axvline(p_fixed, color='gray', linestyle=':', alpha=0.6)
plt.xlabel('q = P(heads) of proposal')
plt.ylabel('KL divergence (nats)')
plt.title('Asymmetry of KL divergence for Bernoulli distributions')
plt.legend()
plt.ylim(0, 3)
plt.tight_layout()
plt.show()

## 3. Cross-Entropy Decomposition

$$H(p, q) = H(p) + D_{\text{KL}}(p \| q)$$

Since $H(p)$ doesn't depend on $q$, minimizing cross-entropy over $q$ is equivalent to minimizing $D_{\text{KL}}(p \| q)$ — this is also maximum likelihood estimation.

In [ ]:
def cross_entropy(p, q):
    """H(p, q) = -sum p log q"""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    return -np.sum(p[mask] * np.log(q[mask]))

test_cases = [
    ([0.5, 0.3, 0.2],  [0.3, 0.4, 0.3]),
    ([0.7, 0.2, 0.1],  [0.1, 0.5, 0.4]),
    ([0.25, 0.25, 0.25, 0.25], [0.4, 0.2, 0.3, 0.1]),
]

print(f"{'H(p)':>10}  {'D_KL(p‖q)':>12}  {'H(p)+KL':>12}  {'H(p,q)':>12}  {'Match':>8}")
print('-' * 65)
for p, q in test_cases:
    h_p   = entropy_discrete(p) * np.log(2)  # convert bits → nats
    kl    = kl_divergence(p, q)
    hpq   = cross_entropy(p, q)
    match = abs((h_p + kl) - hpq) < 1e-10
    print(f"{h_p:>10.6f}  {kl:>12.6f}  {h_p+kl:>12.6f}  {hpq:>12.6f}  {'✓' if match else '✗':>8}")

print()
print("Since H(p) is constant w.r.t. q, minimizing H(p,q) over q = minimizing D_KL(p‖q).")
print("Training with cross-entropy loss IS minimizing forward KL — i.e., MLE.")

## 4. Forward vs Reverse KL Fitting

Given a bimodal target $p$ (mixture of two Gaussians), we fit a single Gaussian $q_\theta = \mathcal{N}(\mu, \sigma^2)$ by minimizing each direction of KL.

- **Forward KL** $D_{\text{KL}}(p \| q)$: $q$ must cover all modes → **mean-seeking** (spreads mass between peaks)
- **Reverse KL** $D_{\text{KL}}(q \| p)$: $q$ can safely ignore modes → **mode-seeking** (collapses onto one peak)

In [ ]:
# Target: mixture of N(-2, 0.7²) and N(2, 0.7²) with equal weights
mix_means = [-2.0, 2.0]
mix_std = 0.7
mix_w = [0.5, 0.5]

def target_pdf(x):
    return sum(w * stats.norm(m, mix_std).pdf(x) for w, m in zip(mix_w, mix_means))

x_grid = np.linspace(-6, 6, 2000)
dx = x_grid[1] - x_grid[0]
p_grid = target_pdf(x_grid)

def forward_kl(params):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    q_grid = stats.norm(mu, sigma).pdf(x_grid) + 1e-300
    mask = p_grid > 1e-300
    return np.sum(p_grid[mask] * np.log(p_grid[mask] / q_grid[mask])) * dx

def reverse_kl(params):
    mu, log_sigma = params
    sigma = np.exp(log_sigma)
    q_grid = stats.norm(mu, sigma).pdf(x_grid) + 1e-300
    p_safe = p_grid + 1e-300
    return np.sum(q_grid * np.log(q_grid / p_safe)) * dx

# Minimize forward KL (mean-seeking)
res_fwd = optimize.minimize(forward_kl, [0.0, 0.0], method='Nelder-Mead')
mu_fwd, sig_fwd = res_fwd.x[0], np.exp(res_fwd.x[1])

# Minimize reverse KL — try both modes as starting points, keep best
best_rev, best_params_rev = np.inf, None
for mu_init in [-2.5, 2.5]:
    res = optimize.minimize(reverse_kl, [mu_init, -0.5], method='Nelder-Mead')
    if res.fun < best_rev:
        best_rev = res.fun
        best_params_rev = res.x
mu_rev, sig_rev = best_params_rev[0], np.exp(best_params_rev[1])

# Plot
plt.figure(figsize=(10, 5))
plt.plot(x_grid, p_grid, 'k', lw=2.5, label='target p (bimodal)')
plt.plot(x_grid, stats.norm(mu_fwd, sig_fwd).pdf(x_grid), 'steelblue', lw=2,
         linestyle='--', label=f'forward KL fit: N({mu_fwd:.2f}, {sig_fwd:.2f}²)  [mean-seeking]')
plt.plot(x_grid, stats.norm(mu_rev, sig_rev).pdf(x_grid), 'coral', lw=2,
         linestyle=':', label=f'reverse KL fit: N({mu_rev:.2f}, {sig_rev:.2f}²)  [mode-seeking]')
plt.xlabel('x')
plt.ylabel('density')
plt.title('Forward KL vs Reverse KL when fitting a Gaussian to a bimodal target')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Forward KL fit: μ={mu_fwd:.3f}, σ={sig_fwd:.3f}  — centers between modes (mean-seeking)")
print(f"Reverse KL fit: μ={mu_rev:.3f}, σ={sig_rev:.3f}  — collapses onto one mode (mode-seeking)")

## 5. Beta Posterior Updates

The Beta distribution $\text{Beta}(\alpha, \beta)$ is the conjugate prior for a Bernoulli likelihood. After observing $h$ heads and $t$ tails, the posterior is:
$$\text{Beta}(\alpha_0 + h,\; \beta_0 + t)$$

Starting from $\text{Beta}(1,1)$ (uniform — maximum uncertainty), we watch the posterior concentrate as coin-flip data accumulates.

In [ ]:
true_p = 0.65  # true coin bias
n_flips = 200
flips = rng.binomial(1, true_p, n_flips)  # 1=heads, 0=tails

theta = np.linspace(0.001, 0.999, 400)
snapshots = [0, 5, 20, 50, 200]

plt.figure(figsize=(10, 5))
colors = plt.cm.Blues(np.linspace(0.3, 0.95, len(snapshots)))

for n, color in zip(snapshots, colors):
    h = flips[:n].sum()
    t = n - h
    alpha, beta = 1 + h, 1 + t
    posterior = stats.beta(alpha, beta).pdf(theta)
    mean = alpha / (alpha + beta)
    label = f'n={n}: Beta({alpha},{beta}), mean={mean:.3f}'
    plt.plot(theta, posterior, color=color, lw=2, label=label)

plt.axvline(true_p, color='red', linestyle='--', lw=1.5, label=f'true p={true_p}')
plt.xlabel('θ (coin bias)')
plt.ylabel('Posterior density')
plt.title('Bayesian updating: Beta posterior as coin-flip data accumulates')
plt.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

# Show pseudo-count interpretation
print("Alpha and beta act as pseudo-counts:")
print(f"  After {n_flips} flips ({flips.sum()} heads, {n_flips-flips.sum()} tails):")
print(f"  Alpha = 1 + {flips.sum()} = {1+flips.sum()}  (prior 1 + observed heads)")
print(f"  Beta  = 1 + {n_flips-flips.sum()} = {1+n_flips-flips.sum()}  (prior 1 + observed tails)")
print(f"  Posterior mean = α/(α+β) = {(1+flips.sum())/(1+n_flips):.4f}  (true: {true_p})")
print(f"  MLE = h/n = {flips.mean():.4f}  — posterior mean approaches MLE as n grows")